# Daily Profile Generation


This notebook converts 15-minute electricity demand data into daily profiles with 96 quarter-hour demand points.

## Steps:
1. Load cleaned data from notebooks/outputs/cleaned_data.csv (made by notebook 01)
2. Create time_slot from Hour and Minute (0-95)
3. Pivot each Date into p00 to p95 columns
4. Remove incomplete days
5. Calculate daily statistics (mean, peak, min, std, etc.)
6. Add temporal features
7. Save daily profiles to notebooks/outputs/daily_profiles.csv


> **Note (19 Oct 2026): this notebook saves its files to `notebooks/outputs/`, not `data/processed/` or `models/`.**
>
> The dashboard and API read `data/processed/` and `models/`. So that running this notebook can never overwrite those production files, every file it saves goes to `notebooks/outputs/` instead (models to `notebooks/outputs/models/`). Notebooks 01 → 05 read each other's files from that folder, so the chain still works end to end.
>
> The logic is the same as the app modules in `backend/app/ml/`. The last cell compares this notebook's results with the production files (read-only).

In [1]:

import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Add project root to Python path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Set up paths
# Outputs go to notebooks/outputs/ so production data/processed/ is never overwritten
DATA_PROCESSED_DIR = PROJECT_ROOT / "notebooks" / "outputs"
CLEANED_DATA_PATH = DATA_PROCESSED_DIR / "cleaned_data.csv"
DAILY_PROFILES_PATH = DATA_PROCESSED_DIR / "daily_profiles.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Cleaned data path: {CLEANED_DATA_PATH}")
print(f"Daily profiles path: {DAILY_PROFILES_PATH}")


Project root: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence
Cleaned data path: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\cleaned_data.csv
Daily profiles path: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\daily_profiles.csv


In [2]:

# Load cleaned data
if not CLEANED_DATA_PATH.exists():
    raise FileNotFoundError(f"Cleaned data not found at {CLEANED_DATA_PATH}")

df_clean = pd.read_csv(CLEANED_DATA_PATH)

print(f"Cleaned data loaded successfully!")
print(f"Shape: {df_clean.shape}")
print(f"Date range: {df_clean['Date'].min()} to {df_clean['Date'].max()}")
print(f"Columns: {list(df_clean.columns)}")
print(f"\nFirst few rows:")
print(df_clean.head())


Cleaned data loaded successfully!
Shape: (189793, 8)
Date range: 2020-01-01 to 2025-05-31
Columns: ['Timestamp', 'Load Demand (kW)', 'Date', 'Hour', 'Minute', 'DayOfWeek', 'Month', 'IsWeekend']

First few rows:
             Timestamp  Load Demand (kW)        Date  Hour  Minute  DayOfWeek  \
0  2020-01-01 00:00:00           1554.47  2020-01-01     0       0          2   
1  2020-01-01 00:15:00           1471.91  2020-01-01     0      15          2   
2  2020-01-01 00:30:00           1581.97  2020-01-01     0      30          2   
3  2020-01-01 00:45:00           1593.76  2020-01-01     0      45          2   
4  2020-01-01 01:00:00           1416.25  2020-01-01     1       0          2   

   Month  IsWeekend  
0      1          0  
1      1          0  
2      1          0  
3      1          0  
4      1          0  


In [3]:

# Create time_slot from Hour and Minute (0-95 for 15-minute intervals)
df_clean['time_slot'] = df_clean['Hour'] * 4 + df_clean['Minute'] // 15

print(f"Time slots created: {df_clean['time_slot'].min()} to {df_clean['time_slot'].max()}")
print(f"Unique time slots: {sorted(df_clean['time_slot'].unique())}")


Time slots created: 0 to 95
Unique time slots: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95]


In [4]:

# Get demand column name (assuming it's the second column after timestamp)
demand_col = df_clean.columns[1]  # Usually the demand/load column
timestamp_col = df_clean.columns[0]  # Usually the timestamp column

print(f"Using demand column: {demand_col}")
print(f"Using timestamp column: {timestamp_col}")

# Pivot to create daily profiles
daily_profiles = df_clean.pivot_table(
    index='Date',
    columns='time_slot',
    values=demand_col,
    aggfunc='mean'
)

# Rename columns to p00, p01, ..., p95
daily_profiles.columns = [f"p{str(col).zfill(2)}" for col in daily_profiles.columns]

print(f"Daily profiles created: {daily_profiles.shape}")
print(f"Profile columns: {list(daily_profiles.columns)}")
print(f"\nFirst few rows:")
print(daily_profiles.head())


Using demand column: Load Demand (kW)
Using timestamp column: Timestamp


Daily profiles created: (1978, 96)
Profile columns: ['p00', 'p01', 'p02', 'p03', 'p04', 'p05', 'p06', 'p07', 'p08', 'p09', 'p10', 'p11', 'p12', 'p13', 'p14', 'p15', 'p16', 'p17', 'p18', 'p19', 'p20', 'p21', 'p22', 'p23', 'p24', 'p25', 'p26', 'p27', 'p28', 'p29', 'p30', 'p31', 'p32', 'p33', 'p34', 'p35', 'p36', 'p37', 'p38', 'p39', 'p40', 'p41', 'p42', 'p43', 'p44', 'p45', 'p46', 'p47', 'p48', 'p49', 'p50', 'p51', 'p52', 'p53', 'p54', 'p55', 'p56', 'p57', 'p58', 'p59', 'p60', 'p61', 'p62', 'p63', 'p64', 'p65', 'p66', 'p67', 'p68', 'p69', 'p70', 'p71', 'p72', 'p73', 'p74', 'p75', 'p76', 'p77', 'p78', 'p79', 'p80', 'p81', 'p82', 'p83', 'p84', 'p85', 'p86', 'p87', 'p88', 'p89', 'p90', 'p91', 'p92', 'p93', 'p94', 'p95']

First few rows:
                p00      p01      p02      p03      p04      p05      p06  \
Date                                                                        
2020-01-01  1554.47  1471.91  1581.97  1593.76  1416.25  1456.35  1544.58   
2020-01-02  1454.50  1474.5

In [5]:

# Remove days with missing time slots (incomplete days)
total_days = len(daily_profiles)
complete_days = daily_profiles.notna().all(axis=1)
daily_profiles = daily_profiles[complete_days]

print(f"Complete days: {complete_days.sum()}/{total_days}")
print(f"Removed {total_days - complete_days.sum()} incomplete days")
print(f"Final daily profiles: {daily_profiles.shape}")

# Fill any remaining missing values with interpolation
daily_profiles = daily_profiles.interpolate(method='linear', axis=1)
daily_profiles = daily_profiles.bfill(axis=1).ffill(axis=1)


Complete days: 1977/1978
Removed 1 incomplete days
Final daily profiles: (1977, 96)


In [6]:

# Calculate daily statistics
profile_cols = [f"p{str(i).zfill(2)}" for i in range(96)]

daily_profiles['daily_mean'] = daily_profiles[profile_cols].mean(axis=1)
daily_profiles['daily_peak'] = daily_profiles[profile_cols].max(axis=1)
daily_profiles['daily_min'] = daily_profiles[profile_cols].min(axis=1)
daily_profiles['daily_std'] = daily_profiles[profile_cols].std(axis=1)
daily_profiles['total_demand'] = daily_profiles[profile_cols].sum(axis=1)

# Find peak time slot
daily_profiles['peak_time_slot'] = daily_profiles[profile_cols].idxmax(axis=1)
daily_profiles['peak_time_slot'] = daily_profiles['peak_time_slot'].str.replace('p', '').astype(int)

print("Daily statistics calculated:")
print(f"- Mean demand: {daily_profiles['daily_mean'].mean():.2f} kW")
print(f"- Peak demand: {daily_profiles['daily_peak'].mean():.2f} kW")
print(f"- Std demand: {daily_profiles['daily_std'].mean():.2f} kW")


Daily statistics calculated:
- Mean demand: 1743.22 kW
- Peak demand: 2591.71 kW
- Std demand: 278.83 kW


In [7]:

# Convert Date to datetime and add temporal features
daily_profiles.index = pd.to_datetime(daily_profiles.index)
daily_profiles['date'] = daily_profiles.index.strftime('%Y-%m-%d')
daily_profiles['day_of_week'] = daily_profiles.index.dayofweek
daily_profiles['month'] = daily_profiles.index.month
daily_profiles['is_weekend'] = (daily_profiles.index.dayofweek.isin([5, 6])).astype(int)

# Reorder columns to have date first, then stats, then profiles
base_cols = ['date', 'daily_mean', 'daily_peak', 'daily_min', 'daily_std', 
            'total_demand', 'peak_time_slot', 'day_of_week', 'month', 'is_weekend']
profile_cols = [f"p{str(i).zfill(2)}" for i in range(96)]
all_cols = base_cols + profile_cols
daily_profiles = daily_profiles[all_cols]

print("Temporal features added:")
print(f"- Date range: {daily_profiles['date'].min()} to {daily_profiles['date'].max()}")
print(f"- Days of week: {sorted(daily_profiles['day_of_week'].unique())}")
print(f"- Months: {sorted(daily_profiles['month'].unique())}")
print(f"- Weekend days: {daily_profiles['is_weekend'].sum()}")


Temporal features added:
- Date range: 2020-01-01 to 2025-05-30
- Days of week: [0, 1, 2, 3, 4, 5, 6]
- Months: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
- Weekend days: 564


In [8]:

# Save daily profiles
daily_profiles.to_csv(DAILY_PROFILES_PATH, index=False)

print(f"Daily profiles saved to: {DAILY_PROFILES_PATH}")
print(f"Final shape: {daily_profiles.shape}")
print(f"File size: {DAILY_PROFILES_PATH.stat().st_size / 1024 / 1024:.1f} MB")

print("\nDaily profiles summary:")
print(f"- Total profiles: {len(daily_profiles)}")
print(f"- Profile columns: {len([col for col in daily_profiles.columns if col.startswith('p')])}")
print(f"- Date range: {daily_profiles['date'].min()} to {daily_profiles['date'].max()}")

print("\nSample of final data:")
print(daily_profiles[['date', 'daily_mean', 'daily_peak', 'daily_std']].head())


Daily profiles saved to: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\daily_profiles.csv
Final shape: (1977, 106)


File size: 1.6 MB

Daily profiles summary:
- Total profiles: 1977
- Profile columns: 97
- Date range: 2020-01-01 to 2025-05-30

Sample of final data:
                  date   daily_mean  daily_peak   daily_std
Date                                                       
2020-01-01  2020-01-01  1712.732396     2518.58  256.266750
2020-01-02  2020-01-02  1712.600938     2552.94  262.234556
2020-01-03  2020-01-03  1713.154375     2531.82  273.122833
2020-01-04  2020-01-04  1412.571979     2121.80  219.954930
2020-01-05  2020-01-05  1401.648021     1993.77  206.009199


In [9]:
# Check against the app's daily profiles (read-only)
app_profiles = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "daily_profiles.csv")

same_dates = list(app_profiles['date']) == list(daily_profiles['date'])
same_profiles = np.allclose(app_profiles[profile_cols].values, daily_profiles[profile_cols].values)
same_peak = np.allclose(app_profiles['peak_demand'].values, daily_profiles['daily_peak'].values)
same_mean = np.allclose(app_profiles['mean_demand'].values, daily_profiles['daily_mean'].values)

print("=== CHECK AGAINST THE APP (data/processed/daily_profiles.csv) ===")
print(f"Days: notebook {len(daily_profiles)} | app {len(app_profiles)} | same dates: {same_dates}")
print(f"Same 96-slot profiles: {same_profiles}")
print(f"Same daily peak / mean: {same_peak} / {same_mean}")

=== CHECK AGAINST THE APP (data/processed/daily_profiles.csv) ===
Days: notebook 1977 | app 1977 | same dates: True
Same 96-slot profiles: True
Same daily peak / mean: True / True
